# Introduction to SageMaker JumpStart - Qwen3.5-27B-FP8




In this demo notebook, we demonstrate how to use the SageMaker Python SDK to deploy a SageMaker JumpStart Qwen3.5-27B-FP8 model and invoke the endpoint.

## Setup
First, upgrade to the latest sagemaker SDK to ensure all available models are deployable.

In [ ]:
%pip install sagemaker==2.257.2 jmespath

Select the desired model to deploy. The provided dropdown filters all text generation models available in SageMaker JumpStart.

In [1]:
model_id = "huggingface-vlm-qwen3-5-27b-fp8"
model_version = "*"

In [2]:
print(model_id)

huggingface-vlm-qwen3-5-27b-fp8


## Deploy model

Create a `JumpStartModel` object, which initializes default model configurations conditioned on the selected instance type. JumpStart already sets a default instance type, but you can deploy the model on other instance types by passing `instance_type` to the `JumpStartModel` class.

In [3]:
from sagemaker.jumpstart.model import JumpStartModel


model = JumpStartModel(model_id=model_id, model_version=model_version)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Using model 'huggingface-vlm-qwen3-5-27b-fp8' with wildcard version identifier '*'. You can pin to version '1.0.1' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


You can now deploy the model using SageMaker JumpStart. If the selected model is gated, you will need to accept the end-user license agreement (EULA) prior to deployment. This is accomplished by providing the `accept_eula=True` argument to the `deploy` method. The deployment might take few minutes. 

In [4]:
predictor = model.deploy()

-----------------!

In [5]:
## If you have an endpoint deployed and you want to use it - copy Endpoint name and uncomment the below lines

# from sagemaker.serializers import JSONSerializer
# from sagemaker.deserializers import JSONDeserializer
# from sagemaker.predictor import Predictor

# predictor = Predictor(
#     endpoint_name="<YOUR ENDPOINT NAME>",
#     serializer=JSONSerializer(),
#     deserializer=JSONDeserializer(),
# )


## Invoke the endpoint

This section demonstrates how to invoke the endpoint using example payloads that are retrieved programmatically from the `JumpStartModel` object. You can replace these example payloads with your own payloads.

In [6]:
example_payloads = model.retrieve_all_examples()

In [7]:
example_payloads[0].body

{'max_tokens': 256,
 'messages': [{'content': 'What is deep learning?', 'role': 'user'}]}

In [8]:
import jmespath


for payload in example_payloads:
    response = predictor.predict(payload.body)
    generated_text = jmespath.search(payload.raw_payload["output_keys"]["generated_text"], response)
    print("Input:\n", payload.body[payload.prompt_key])
    print("Output:\n", generated_text.strip())
    print("\n===============\n")

Input:
 [{'content': 'What is deep learning?', 'role': 'user'}]
Output:
 Here's a thinking process that leads to the explanation of deep learning:

1.  **Deconstruct the Request:**
    *   **Question:** "What is deep learning?"
    *   **Intent:** The user wants a clear, comprehensive, yet accessible definition and explanation of deep learning. They might be a beginner, a student, or someone curious about AI trends.
    *   **Key Concepts to Cover:** Definition, relation to AI/ML, neural networks, "deep" meaning, how it works (briefly), applications, pros/cons.

2.  **Initial Brainstorming & Structuring:**
    *   *Analogy:* How do I explain this simply? (Brain/Neurons).
    *   *Hierarchy:* AI > Machine Learning > Deep Learning.
    *   *Core Mechanism:* Artificial Neural Networks (ANNs), layers, weights, backpropagation.
    *   *Why "Deep"?* Many hidden layers.
    *   *Use Cases:* Images, text, speech, games.
    *   *Requirements:* Data, compute power.
    *   *Structure:*
       

## Invoke the endpoint with no reasoning

In [15]:
payload = {
    "messages": [
        {"role": "user", "content": "Hi, tell me a joke"}
    ],
    "max_tokens": 128,
    "temperature": 0.6,
    "chat_template_kwargs": {"enable_thinking": False}
}

response = predictor.predict(payload)
print(response['choices'][0]['message']['content'])


Why don't scientists trust atoms?

Because they **make up everything**! 😄


## Invoke the endpoint with a dummy Hebrew CV

In [16]:
import base64
import json
import boto3

# Encode your image
with open("dummy_cv_heb_1.png", "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")

payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_base64}"
                    }
                },
                {
                    "type": "text",
                    # "text": "Extract name, title, experience and other text fields of the candidate from this CV. /no_think"
                    "text": "Convert this document into Markdown while preserving the structure /no_think" 
                }
            ]
        }
    ],
    "max_tokens": 1024,
    "temperature": 0.0,
    "chat_template_kwargs": {"enable_thinking": False}
}

response = predictor.predict(payload)
print(response)


{'id': 'chatcmpl-b84c3115f17bbc1d', 'object': 'chat.completion', 'created': 1779892900, 'model': '/opt/ml/model', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "# איתי לוי\n## Full Stack מפתח\n\nמפתח תוכנה עם 4+ שנות ניסיון בפיתוח יישומי Web מאפס ועד פרודקשן.\nמתמחה ב-React, Node.js, TypeScript, React Native ו-Python.\nאוהב לפתור בעיות מורכבות, כותב קוד נקי, עובד היטב בצוות\nומתמיד בלמידה של טכנולוגיות חדשות.\n\n---\n\n### פרטים אישיים\n- 📧 itai.levi.dev@gmail.com\n- 📞 054-1234567\n- 📍 תל אביב, ישראל\n- 🔗 linkedin.com/in/itai-levi-dev\n- 🐙 github.com/itai-levi\n\n---\n\n### ניסיון תעסוקתי\n\n#### מפתח Full Stack\n**TechNova**\nינואר 2022 | תל אביב\n- פיתוח ותחזוקה של אפליקציות Web בסביבת React, Node.js ו-MongoDB\n- עבודה עם טכנולוגיות מודרניות כמו TypeScript ו-Next.js\n- שילוב ביצועים ואופטימיזציה של זמני טעינה\n- הובלת תהליכי Code Review ונוכחות מפתחים חדשים\n\n#### מפתח Full Stack\n**DataWave**\nיוני 2020 - דצמבר 2021 | רמת גן\n- פיתוח פיצ'רים חדשים ושופור מערכ

In [17]:
print(response['choices'][0]['message']['content'])

# איתי לוי
## Full Stack מפתח

מפתח תוכנה עם 4+ שנות ניסיון בפיתוח יישומי Web מאפס ועד פרודקשן.
מתמחה ב-React, Node.js, TypeScript, React Native ו-Python.
אוהב לפתור בעיות מורכבות, כותב קוד נקי, עובד היטב בצוות
ומתמיד בלמידה של טכנולוגיות חדשות.

---

### פרטים אישיים
- 📧 itai.levi.dev@gmail.com
- 📞 054-1234567
- 📍 תל אביב, ישראל
- 🔗 linkedin.com/in/itai-levi-dev
- 🐙 github.com/itai-levi

---

### ניסיון תעסוקתי

#### מפתח Full Stack
**TechNova**
ינואר 2022 | תל אביב
- פיתוח ותחזוקה של אפליקציות Web בסביבת React, Node.js ו-MongoDB
- עבודה עם טכנולוגיות מודרניות כמו TypeScript ו-Next.js
- שילוב ביצועים ואופטימיזציה של זמני טעינה
- הובלת תהליכי Code Review ונוכחות מפתחים חדשים

#### מפתח Full Stack
**DataWave**
יוני 2020 - דצמבר 2021 | רמת גן
- פיתוח פיצ'רים חדשים ושופור מערכות קיימות
- עבודה עם Express.js, PostgreSQL ו-Redux
- בניית ממשקי משתמש רספונסיביים ב-React ו-Redux
- כתיבת בדיקות אוטומטיות עם Jest ו-Cypress

#### מפתח תוכנה
**SoftLogic**
אוקטובר 2019 - מאי 2020 | ירושלים
- פיתוח 

In [18]:
from IPython.display import Markdown, display

display(Markdown(response['choices'][0]['message']['content']))

# איתי לוי
## Full Stack מפתח

מפתח תוכנה עם 4+ שנות ניסיון בפיתוח יישומי Web מאפס ועד פרודקשן.
מתמחה ב-React, Node.js, TypeScript, React Native ו-Python.
אוהב לפתור בעיות מורכבות, כותב קוד נקי, עובד היטב בצוות
ומתמיד בלמידה של טכנולוגיות חדשות.

---

### פרטים אישיים
- 📧 itai.levi.dev@gmail.com
- 📞 054-1234567
- 📍 תל אביב, ישראל
- 🔗 linkedin.com/in/itai-levi-dev
- 🐙 github.com/itai-levi

---

### ניסיון תעסוקתי

#### מפתח Full Stack
**TechNova**
ינואר 2022 | תל אביב
- פיתוח ותחזוקה של אפליקציות Web בסביבת React, Node.js ו-MongoDB
- עבודה עם טכנולוגיות מודרניות כמו TypeScript ו-Next.js
- שילוב ביצועים ואופטימיזציה של זמני טעינה
- הובלת תהליכי Code Review ונוכחות מפתחים חדשים

#### מפתח Full Stack
**DataWave**
יוני 2020 - דצמבר 2021 | רמת גן
- פיתוח פיצ'רים חדשים ושופור מערכות קיימות
- עבודה עם Express.js, PostgreSQL ו-Redux
- בניית ממשקי משתמש רספונסיביים ב-React ו-Redux
- כתיבת בדיקות אוטומטיות עם Jest ו-Cypress

#### מפתח תוכנה
**SoftLogic**
אוקטובר 2019 - מאי 2020 | ירושלים
- פיתוח מודולים ב-Python (Django)
- עבודה עם מסדי נתונים וכתיבת שאילתות SQL
- תיקון באגים ושופור תהליכי CI/CD
- שיתוף פעולה עם צוותי QA וטוענים

---

### מימנויות
- JavaScript / TypeScript
- React / Next.js
- Node.js / Express
- Python / Django
- SQL / NoSQL
- HTML / CSS / SASS
- Git / GitHub
- Docker

---

### כלים וטכנולוגיות
- React
- Next.js
- Node.js
- TypeScript
- Python
- Django
- PostgreSQL
- MongoDB
- Redis
- Docker
- Git
- Jest
- Cypress
- AWS
- CI/CD
- GraphQL

---

### פרויקטים אישיים

#### DevBlog
פלטפורמה לבלוגים למפתחים עם מערכת ניהול תוכן.
טכנולוגיות

## Clean up the endpoint
Don't forget to clean up resources when finished to avoid unnecessary charges.

In [ ]:
predictor.delete_predictor()